# Brain Tumor Segmentation *BraTS&mdash;Style*
This notebook will implement a *BraTS-Style* Lighting pipeline that will train a model to preform brain tumor segmentation.  The trained model will then be used to segment ***post-operative*** brain tumor MRI data.  Below is the data directory structure that will be used for this study.
```
Brats2020/
├── training/                                              # Has both images & segmentation
│   ├── BRATS_001/
│   │   ├── BRATS_001_t1.nii.gz
│   │   ├── BRATS_001_t1ce.nii.gz
│   │   ├── BRATS_001_t2.nii.gz
│   │   ├── BRATS_001_flair.nii.gz
│   │   └── BRATS_001_seg.nii.gz                           # segmentation
│   ├── BRATS_002/ ..
│   └── ...
├── validation/                                            # Only 4 modalities (no seg)
│   ├── BRATS_001/
│   │   ├── BRATS_001_t1.nii.gz
│   │   ├── ...
│   │   └── BRATS_001_flair.nii.gz
│   └── ...
└── ...
```
- Data loading, transformations, and model training metrix collections will be accomplished using the **MONAI** package. ***MONAI*** is optimized for MRI-specific transforms, loss functions, and evaluation metrics for medical imaging

----
<a name='setup_environment'></a>
## 1.0 <span style='color:blue'>|</span> Setup Environment

<a name='load_packages'></a>
### 1.1 <span style='color:blue'>|</span> Load Required Libraries and Packages

In [1]:
import os, glob, re, random
import numpy as np
import nibabel as nip
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0' # Fixes a warning from Pytorch
import torch, monai
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from monai.transforms import (
    LoadImaged, EnsureChannelFirstd, ScaleIntensityRanged, CropForegroundd,
    ResizeWithPadOrCropd, Compose, ToTensord, EnsureTyped, Spacingd
)
from monai.networks.nets import UNet
from monai.losses import DiceLoss
from monai.metrics import compute_dice

from pathlib import Path
from typing import List, Tuple, Dict

# Squash Python warnings
import warnings
warnings.filterwarnings('ignore')

# Enable Python's Garbage Collector
import gc
gc.collect()

<frozen importlib._bootstrap_external>:1325: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
2026-02-21 12:09:08.759103: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


418

<a name='global_variables'></a>
### 1.2 <span style='color:blue'>|</span> Declare Global Variables

In [4]:
#DATA_DIR = '../BraTS/'
#TRN_DIR = DATA_DIR+'BraTS2021_TrainingSet/UPENN-GBM/'
#VAL_DIR = DATA_DIR+'BraTS2021_ValidationSet/UPENN-GBM/'

seed = 42

<a name='set_randomseed'></a>
### 1.3 <span style='color:blue'>|</span> Set Random Seed for Reproducibility

In [5]:
# Set random seed for reproducibility
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
os.environ['PYTHONHASHSEED'] = str(seed)

# When running on CuDNN backend, it is recommended to set these two options
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.set_float32_matmul_precision('medium')

# Make sure GPU is used, if available
# Do this in some of the classes and functions, but want it to be global also
device = 'cuda' if torch.cuda.is_available() else 'cpu'

<a name='load_datasets'></a>
### 1.4 <span style='color:blue'>|</span> Recursively Glob for BraTS-style Paths
Recursively transverse the data structure and load NIfTI files, transform them and return the image along with its label 

In [21]:
def find_brats_cases(root_dir: str) -> List[Dict[str, str]]:
    '''
    Recursively find all BraTS cases (t1, t1ce, t2, flair, seg).
    Returns list of dicts: {t1: path, t1ce: path, ..., seg: path or None}
    '''
    root = Path(root_dir)
    training_dir = root / 'BraTS2021_TrainingSet/UPENN-GBM/'
    print(training_dir)
    val_dir = root / 'BraTS2021_ValidationSet/UPENN-GBM/'
    
    cases = []

    def process_dir(path):
        if not path.exists():
            return
        # Find subdirectories (case folders)
        for case_folder in sorted(path.glob('BraTS2021_*')):
            if not case_folder.is_dir():
                continue

            case_data = {'case_name': case_folder.name, 'modality_paths': {}}
            # Look for modality files
            for mod in ['t1', 't1ce', 't2', 'flair', 'seg']:
                pattern = case_folder / f'{case_folder.name}_{mod}.nii*'
                files = list(glob.glob(str(pattern)))  # supports .nii or .nii.gz
                if files:
                    case_data['modality_paths'][mod] = files[0]
                elif mod == 'seg':
                    # For val set, seg should be missing → we mark it None explicitly
                    case_data['modality_paths'][mod] = None
                else:
                    raise FileNotFoundError(f'Missing modality {mod} in {case_folder}')
            cases.append(case_data)

    if training_dir.exists():
        process_dir(training_dir)
    if val_dir.exists():
        process_dir(val_dir)

    # Optional: Validate each case has at least 4 modalities
    cases = [c for c in cases if len(c['modality_paths']) >= 4]
    print(f'Found {len(cases)} cases with full modalities across {root_dir}')
    return cases


In [22]:
cases = find_brats_cases('~/regis/dataScience/segmentation/BraTS')

~/regis/dataScience/segmentation/BraTS/BraTS2021_TrainingSet/UPENN-GBM
Found 0 cases with full modalities across ~/regis/dataScience/segmentation/BraTS


In [ ]:
class BraTSDataset(Dataset):
    def __init__(
        self,
        cases: List[Dict[str, str]],
        transform: Compose,
        train_mode: bool = True,  # if True, expects 'seg' to be present
        image_size: Tuple[int, int, int] = (128, 128, 128),
    ):
        self.cases = cases
        self.transform = transform
        self.train_mode = train_mode

        # Validate: if train_mode=True, all cases must have seg
        if train_mode and not all(c['modality_paths']['seg'] is not None for c in cases):
            raise ValueError('Training dataset requires segmentation labels for all cases.')

    def __len__(self):
        return len(self.cases)

    def __getitem__(self, idx):
        case = self.cases[idx]
        inputs = [case['modality_paths'][mod] for mod in ['t1', 't1ce', 't2', 'flair']]
        target_path = case['modality_paths']['seg'] if self.train_mode else None

        data_dict = {
            't1': inputs[0],
            't1ce': inputs[1],
            't2': inputs[2],
            'flair': inputs[3],
        }
        if self.train_mode and target_path:
            data_dict['seg'] = target_path

        # Apply transforms ( LoadImaged --> EnsureChannelFirstd --> ... )
        # Note: transform expects filenames in keys
        if self.transform:
            data_dict = self.transform(data_dict)

        # Combine modalities into single tensor (C, H, W, D)
        x = torch.stack([
            data_dict['t1'],
            data_dict['t1ce'],
            data_dict['t2'],
            data_dict['flair'],
        ], dim=0)  # shape: [4, H, W, D]

        y = None
        if self.train_mode:
            # Segmentation: combine ET, TC, WT labels into 3-class (or 1 for whole tumor)
            # BraTS 2020+ uses multi-class (0, 1, 2, 4) --> we usually want ET (4), WT (1+2+4), TC (1+4)
            # Simplify to 2-class (0=background, 1=tumor) or keep 3-class if desired
            y = data_dict['seg']  # shape: [1, H, W, D]
            # Optional: post-process seg (e.g., relabel for 3-class)
            # Example: whole tumor = label 1 (0,1,2,4 → 0 vs ≥1)
            # y = (y > 0).float()

        return {'image': x, 'label': y}

In [ ]:
class DataModule(LightningDataModule):
    def __init__(self, train_dir, val_dir=None, batch_size=2, num_workers=4, patch_size=(128, 128, 128)):
        super().__init__()
        self.train_dir = train_dir
        self.val_dir = val_dir
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.patch_size = patch_size

        # Preprocessing (CPU/GPU compatible)
        self.preprocessing = tio.Compose([
            tio.Resample((1.0, 1.0, 1.0)),
            tio.ToCanonical(),
            tio.CropOrPad(self.patch_size, padding_mode=0),
            tio.ZNormalization(masking_method=tio.ZNormalization.MEAN),
        ])

    def setup(self, stage=None):
        # Load patients
        subjects_train = self._load_subjects(self.train_dir, stage='train')
        self.train_ds = tio.SubjectsDataset(subjects_train, transform=self.preprocessing)

        if self.val_dir:
            subjects_val = self._load_subjects(self.val_dir, stage='val')
            self.val_ds = tio.SubjectsDataset(subjects_val, transform=self.preprocessing)

    def _load_subjects(self, directory, stage='train'):
        subjects = []
        for patient_dir in sorted(os.listdir(directory)):
            patient_path = os.path.join(directory, patient_dir)
            if not os.path.isdir(patient_path):
                continue

            # Expect 4 modalities: T1, T1ce, T2, FLAIR (case-insensitive)
            flair = self._find_modality(patient_path, ['flair'])
            t1 = self._find_modality(patient_path, ['t1', 't1n', 't1-weighted'])
            t1ce = self._find_modality(patient_path, ['t1ce', 't1c', 't1-gad'])
            t2 = self._find_modality(patient_path, ['t2', 't2-weighted'])

            if not all([flair, t1, t1ce, t2]):
                continue

            # Load label (BraTS format: 0,1,2,4 → remapped to [0,1,2,3])
            label_path = self._find_label(patient_path)
            if stage != 'predict' and not label_path:
                continue  # skip patients without labels in train/val

            # Create subject
            subject_data = {
                'flair': tio.Image(flair, type=tio.INTENSITY),
                't1': tio.Image(t1, type=tio.INTENSITY),
                't1ce': tio.Image(t1ce, type=tio.INTENSITY),
                't2': tio.Image(t2, type=tio.INTENSITY),
            }

            if label_path:
                label_img = nib.load(label_path).get_fdata()
                # Remap: 1/4→1, 2→2, 0→0, 3→1 (as per BraTS)
                label_map = np.zeros_like(label_img, dtype=np.uint8)
                label_map[label_img == 1] = 1  # necrosis
                label_map[label_img == 2] = 2  # edema
                label_map[np.isin(label_img, [3, 4])] = 1  # enhancing & active tumor
                label_img = torch.from_numpy(label_map)

                subject_data['label'] = tio.LabelMap(tensor=label_img, type=tio.LABEL)

            subject = tio.Subject(subject_data)
            subjects.append(subject)

        return subjects

    def _find_modality(self, path, patterns):
        for f in os.listdir(path):
            if any(p in f.lower() for p in patterns):
                return os.path.join(path, f)
        return None

    def _find_label(self, path):
        for f in os.listdir(path):
            if 'seg' in f.lower():
                return os.path.join(path, f)
        return None

    def train_dataloader(self):
        return DataLoader(
            self.train_ds,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            pin_memory=True,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_ds,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True,
        )

In [ ]:
class LightningModule(pl.LightningModule):
    def __init__(self, learning_rate=1e-4):
        super().__init__()
        self.save_hyperparameters()
        self.model = UNet3D(in_channels=4, out_channels=3)
        self.dice = Dice(num_classes=3, average='macro', mdmc_average='global')

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, 'train')

    def validation_step(self, batch, batch_idx):
        return self._shared_step(batch, 'val')

    def _shared_step(self, batch, stage):
        image, label = batch['image'], batch['label']  # image: [B,4,H,W,D], label: [B,3,H,W,D]
        pred = self(image)
        loss = F.cross_entropy(pred, torch.argmax(label, dim=1))  # label is one-hot
        dice = self.dice(pred.softmax(dim=1), label.argmax(dim=1))

        self.log(f'{stage}_loss', loss, prog_bar=True)
        self.log(f'{stage}_dice', dice, prog_bar=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)
        return {'optimizer': optimizer}

In [ ]:
def main():
    # Paths (adjust to your setup)
    train_dir = '/path/to/BraTS20/train'
    val_dir = '/path/to/BraTS20/val'

    data_module = DataModule(
        train_dir=train_dir, val_dir=val_dir,
        batch_size=2, num_workers=4,
        patch_size=(128, 128, 128)
    )

    model = LightningModule(learning_rate=1e-4)

    trainer = pl.Trainer(
        max_epochs=100,
        accelerator='gpu',
        devices=1,
        logger=True,
        log_every_n_steps=10,
        enable_checkpointing=True,
        precision=16,  # mixed precision for faster training
    )

    trainer.fit(model, data_module)

if __name__ == '__main__':
    main()